# 🎯 Adult Census Income — AutoML vs Manual Verification
### Dataset: `adult.csv` (32,561 rows × 15 columns)
### Task: **Binary Classification** — Predict income `<=50K` or `>50K`

| Column | Type | Unique | Description |
|--------|------|--------|-------------|
| age | int | 73 | Age of individual |
| workclass | cat | 9 | Type of employment |
| fnlwgt | int | 21648 | Census weight |
| education | cat | 16 | Education level |
| education.num | int | 16 | Education numeric |
| marital.status | cat | 7 | Marital status |
| occupation | cat | 15 | Job type |
| relationship | cat | 6 | Family relationship |
| race | cat | 5 | Race |
| sex | cat | 2 | Gender (binary) |
| capital.gain | int | 119 | Capital gains |
| capital.loss | int | 92 | Capital losses |
| hours.per.week | int | 94 | Weekly work hours |
| native.country | cat | 42 | Country of origin |
| **income** | **target** | **2** | **<=50K / >50K** |

## 1️⃣ Import Libraries & Load Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle

from sklearn.preprocessing import LabelEncoder, RobustScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All libraries imported")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Load adult.csv
DATA_PATH = r"backend\storage\users\d4f48123-b254-47f8-a335-6d142b3f1a72\files\adult.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "adult.csv"  # fallback

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

## 2️⃣ Data Overview & Missing Values

In [ ]:
# Dataset info
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nBasic stats:")
df.describe()

In [ ]:
# Check '?' values (this dataset uses '?' instead of NaN)
print("Columns with '?' values:")
for col in df.select_dtypes(include='object').columns:
    q_count = (df[col].str.strip() == '?').sum()
    if q_count > 0:
        print(f"  {col}: {q_count} rows ({q_count/len(df)*100:.1f}%)")

# Check unique values for each categorical column
print("\nCategorical columns - unique values:")
for col in df.select_dtypes(include='object').columns:
    print(f"  {col} ({df[col].nunique()}): {df[col].unique()[:10]}")

## 3️⃣ EDA — Exploratory Data Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(x='income', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Income Distribution (Target)')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}\n({p.get_height()/len(df)*100:.1f}%)',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom')

# Pie chart
df['income'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1], colors=['#66b3ff','#ff9999'])
axes[1].set_title('Income Split')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()
print(f"\nClass imbalance: <=50K={24720} ({24720/32561*100:.1f}%), >50K={7841} ({7841/32561*100:.1f}%)")

In [ ]:
# Numeric columns distribution
numeric_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, hue='income', kde=True, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} by Income')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical columns vs Income
categorical_cols = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex']

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    ct = pd.crosstab(df[col], df['income'], normalize='index') * 100
    ct.plot(kind='barh', stacked=True, ax=axes[i], color=['#66b3ff','#ff9999'])
    axes[i].set_title(f'{col} vs Income (%)')
    axes[i].set_xlabel('Percentage')

axes[7].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax)
ax.set_title('Numeric Features Correlation')
plt.tight_layout()
plt.show()

## 4️⃣ Data Cleaning
- Replace `?` with NaN, then fill with mode (most frequent value)
- Strip whitespace from string columns
- Remove duplicate rows

In [ ]:
# Correlation heatmap of numeric features
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax)
ax.set_title('Numeric Features Correlation')
plt.tight_layout()
plt.show()

## 5️⃣ Feature Engineering
- **Target**: LabelEncode `income` → `<=50K`=0, `>50K`=1
- **Binary** (`sex`): LabelEncode → Male=1, Female=0
- **Low cardinality** (`workclass`, `education`, `marital.status`, `occupation`, `relationship`, `race`): One-Hot Encoding
- **High cardinality** (`native.country` - 42 unique): Label + Frequency Encoding
- **Numeric** (`age`, `fnlwgt`, `education.num`, `capital.gain`, `capital.loss`, `hours.per.week`): RobustScaler

In [ ]:
# ============================
# ENCODE TARGET: income
# ============================
target_encoder = LabelEncoder()
df_clean['income_encoded'] = target_encoder.fit_transform(df_clean['income'])
print(f"Target encoded: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}")
print(f"  <=50K → 0, >50K → 1")

# Separate target
y = df_clean['income_encoded'].values
print(f"\ny distribution: {np.unique(y, return_counts=True)}")

# ============================
# DEFINE COLUMN GROUPS
# ============================
numeric_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
binary_cols = ['sex']                           # 2 unique → Label Encode
onehot_cols = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race']  # 5-16 unique → One-Hot
highcard_cols = ['native.country']              # 42 unique → Label + Frequency

print(f"\nNumeric  ({len(numeric_cols)}): {numeric_cols}")
print(f"Binary   ({len(binary_cols)}): {binary_cols}")
print(f"One-Hot  ({len(onehot_cols)}): {onehot_cols}")
print(f"HighCard ({len(highcard_cols)}): {highcard_cols}")

In [ ]:
# ============================
# NUMERIC: RobustScaler
# ============================
scaler = RobustScaler()
X_numeric = scaler.fit_transform(df_clean[numeric_cols])
print(f"Numeric features: {X_numeric.shape} — {numeric_cols}")

# ============================
# BINARY: LabelEncode sex
# ============================
sex_encoder = LabelEncoder()
X_sex = sex_encoder.fit_transform(df_clean['sex']).reshape(-1, 1)
print(f"Binary features:  {X_sex.shape} — sex ({list(sex_encoder.classes_)})")

# ============================
# ONE-HOT: workclass, education, marital.status, occupation, relationship, race
# ============================
X_onehot = pd.get_dummies(df_clean[onehot_cols], drop_first=False)
onehot_feature_names = X_onehot.columns.tolist()
X_onehot = X_onehot.values.astype(float)
print(f"One-Hot features: {X_onehot.shape} — from {onehot_cols}")

# ============================
# HIGH CARDINALITY: native.country → Label + Frequency
# ============================
country_encoder = LabelEncoder()
X_country_label = country_encoder.fit_transform(df_clean['native.country']).reshape(-1, 1).astype(float)

country_freq = df_clean['native.country'].value_counts(normalize=True).to_dict()
X_country_freq = df_clean['native.country'].map(country_freq).values.reshape(-1, 1).astype(float)

X_highcard = np.hstack([X_country_label, X_country_freq])
print(f"HighCard features: {X_highcard.shape} — native.country (label + frequency)")

# ============================
# COMBINE ALL FEATURES
# ============================
X = np.hstack([X_numeric, X_sex, X_onehot, X_highcard])

feature_names = (
    numeric_cols + 
    ['sex_binary'] + 
    onehot_feature_names + 
    ['native.country_encoded', 'native.country_freq']
)

print(f"\n✅ Final feature matrix: {X.shape}")
print(f"   Total features: {len(feature_names)}")
print(f"   Samples: {X.shape[0]}")
print(f"   Target classes: {list(target_encoder.classes_)}")

## 6️⃣ Train / Test Split (80/20, Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"Features:  {X_train.shape[1]}")
print(f"\nTrain target distribution: {np.unique(y_train, return_counts=True)}")
print(f"Test target distribution:  {np.unique(y_test, return_counts=True)}")

## 7️⃣ Train Models
Training 3 classifiers with **same parameters as AutoML**:
1. **Random Forest** — `n_estimators=150, max_depth=10, class_weight='balanced'`
2. **XGBoost** — `n_estimators=150, max_depth=6, learning_rate=0.1`
3. **LightGBM** — `n_estimators=150, max_depth=10, learning_rate=0.1`

In [ ]:
from time import time

# ========================================
# MODEL 1: Random Forest (Same as AutoML)
# ========================================
print("=" * 50)
print("MODEL 1: Random Forest")
print("=" * 50)

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

t0 = time()
rf_model.fit(X_train, y_train)
rf_time = time() - t0

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')
rf_prec = precision_score(y_test, rf_pred, average='weighted')
rf_rec = recall_score(y_test, rf_pred, average='weighted')
rf_auc = roc_auc_score(y_test, rf_proba)

print(f"  Training time: {rf_time:.2f}s")
print(f"  Accuracy:      {rf_acc:.4f} ({rf_acc*100:.2f}%)")
print(f"  F1-Score:      {rf_f1:.4f}")
print(f"  Precision:     {rf_prec:.4f}")
print(f"  Recall:        {rf_rec:.4f}")
print(f"  AUC-ROC:       {rf_auc:.4f}")
print(f"\n{classification_report(y_test, rf_pred, target_names=list(target_encoder.classes_))}")

In [ ]:
# ========================================
# MODEL 2: XGBoost
# ========================================
print("=" * 50)
print("MODEL 2: XGBoost")
print("=" * 50)

try:
    from xgboost import XGBClassifier
    
    xgb_model = XGBClassifier(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        use_label_encoder=False
    )
    
    t0 = time()
    xgb_model.fit(X_train, y_train)
    xgb_time = time() - t0
    
    xgb_pred = xgb_model.predict(X_test)
    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
    
    xgb_acc = accuracy_score(y_test, xgb_pred)
    xgb_f1 = f1_score(y_test, xgb_pred, average='weighted')
    xgb_prec = precision_score(y_test, xgb_pred, average='weighted')
    xgb_rec = recall_score(y_test, xgb_pred, average='weighted')
    xgb_auc = roc_auc_score(y_test, xgb_proba)
    
    print(f"  Training time: {xgb_time:.2f}s")
    print(f"  Accuracy:      {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")
    print(f"  F1-Score:      {xgb_f1:.4f}")
    print(f"  Precision:     {xgb_prec:.4f}")
    print(f"  Recall:        {xgb_rec:.4f}")
    print(f"  AUC-ROC:       {xgb_auc:.4f}")
    print(f"\n{classification_report(y_test, xgb_pred, target_names=list(target_encoder.classes_))}")
except ImportError:
    print("  ⚠️ XGBoost not installed. Run: pip install xgboost")
    xgb_acc = xgb_f1 = xgb_prec = xgb_rec = xgb_auc = xgb_time = 0

In [ ]:
# ========================================
# MODEL 3: LightGBM
# ========================================
print("=" * 50)
print("MODEL 3: LightGBM")
print("=" * 50)

try:
    from lightgbm import LGBMClassifier
    
    lgbm_model = LGBMClassifier(
        n_estimators=150,
        max_depth=10,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    t0 = time()
    lgbm_model.fit(X_train, y_train)
    lgbm_time = time() - t0
    
    lgbm_pred = lgbm_model.predict(X_test)
    lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
    
    lgbm_acc = accuracy_score(y_test, lgbm_pred)
    lgbm_f1 = f1_score(y_test, lgbm_pred, average='weighted')
    lgbm_prec = precision_score(y_test, lgbm_pred, average='weighted')
    lgbm_rec = recall_score(y_test, lgbm_pred, average='weighted')
    lgbm_auc = roc_auc_score(y_test, lgbm_proba)
    
    print(f"  Training time: {lgbm_time:.2f}s")
    print(f"  Accuracy:      {lgbm_acc:.4f} ({lgbm_acc*100:.2f}%)")
    print(f"  F1-Score:      {lgbm_f1:.4f}")
    print(f"  Precision:     {lgbm_prec:.4f}")
    print(f"  Recall:        {lgbm_rec:.4f}")
    print(f"  AUC-ROC:       {lgbm_auc:.4f}")
    print(f"\n{classification_report(y_test, lgbm_pred, target_names=list(target_encoder.classes_))}")
except ImportError:
    print("  ⚠️ LightGBM not installed. Run: pip install lightgbm")
    lgbm_acc = lgbm_f1 = lgbm_prec = lgbm_rec = lgbm_auc = lgbm_time = 0

## 8️⃣ Model Comparison — All 3 Models Side by Side

In [ ]:
# Comparison table
results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'LightGBM'],
    'Accuracy': [rf_acc, xgb_acc, lgbm_acc],
    'F1-Score': [rf_f1, xgb_f1, lgbm_f1],
    'Precision': [rf_prec, xgb_prec, lgbm_prec],
    'Recall': [rf_rec, xgb_rec, lgbm_rec],
    'AUC-ROC': [rf_auc, xgb_auc, lgbm_auc],
    'Time (s)': [rf_time, xgb_time, lgbm_time]
})

# Highlight best
print("=" * 70)
print("  MODEL COMPARISON — Manual Training Results")
print("=" * 70)
print(results.to_string(index=False, float_format='%.4f'))

best_idx = results['Accuracy'].idxmax()
print(f"\n🏆 Best Model: {results.loc[best_idx, 'Model']} (Accuracy: {results.loc[best_idx, 'Accuracy']:.4f})")

# Bar chart comparison
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics = ['Accuracy', 'F1-Score', 'Precision', 'AUC-ROC']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, metric in enumerate(metrics):
    bars = axes[i].bar(results['Model'], results[metric], color=colors)
    axes[i].set_title(metric)
    axes[i].set_ylim(0.7, 1.0)
    for bar, val in zip(bars, results[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Manual Model Comparison — adult.csv', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all 3 models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_names = list(target_encoder.classes_)

for i, (name, y_p) in enumerate([('Random Forest', rf_pred), ('XGBoost', xgb_pred), ('LightGBM', lgbm_pred)]):
    cm = confusion_matrix(y_test, y_p)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=axes[i], cmap='Blues')
    axes[i].set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_p):.4f}')

plt.suptitle('Confusion Matrices — Manual Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9️⃣ Feature Importance — Top 20 Features

In [ ]:
# Feature importance from Random Forest
importances = rf_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Top 20
top_20 = feat_imp.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top_20, x='Importance', y='Feature', palette='viridis', ax=ax)
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=14)
ax.set_xlabel('Importance')

for i, (_, row) in enumerate(top_20.iterrows()):
    ax.text(row['Importance'] + 0.001, i, f"{row['Importance']:.4f}", va='center')

plt.tight_layout()
plt.show()

print("\nAll feature importances:")
print(feat_imp.to_string(index=False))

## 🔟 Load AutoML .pkl & Compare with Manual Results

In [ ]:
# Load AutoML saved model
PKL_PATH = r"backend\storage\automl\d4f48123-b254-47f8-a335-6d142b3f1a72\model.pkl"
if not os.path.exists(PKL_PATH):
    PKL_PATH = "model.pkl"  # fallback

if os.path.exists(PKL_PATH):
    with open(PKL_PATH, 'rb') as f:
        automl_data = pickle.load(f)
    
    print("✅ AutoML .pkl loaded successfully!")
    print(f"   Keys: {list(automl_data.keys())}")
    print(f"   Model: {automl_data.get('model_name', 'Unknown')}")
    print(f"   Task:  {automl_data.get('task_type', 'Unknown')}")
    print(f"   Target: {automl_data.get('target_column', 'Unknown')}")
    
    # Extract AutoML metrics
    automl_metrics = automl_data.get('metrics', {})
    print(f"\n   AutoML Saved Metrics:")
    for k, v in automl_metrics.items():
        if isinstance(v, (int, float)):
            print(f"     {k}: {v:.4f}")
else:
    print(f"⚠️ No AutoML .pkl found at: {PKL_PATH}")
    print("   Train a model via AutoML first, then download the .pkl file")
    automl_data = None

In [ ]:
# ========================================
# FINAL COMPARISON: AutoML vs Manual
# ========================================
if automl_data is not None:
    automl_metrics = automl_data.get('metrics', {})
    automl_model_name = automl_data.get('model_name', 'Unknown')
    
    automl_acc = automl_metrics.get('accuracy', 0)
    automl_f1 = automl_metrics.get('f1', automl_metrics.get('f1_score', 0))
    automl_prec = automl_metrics.get('precision', 0)
    automl_rec = automl_metrics.get('recall', 0)
    
    print("=" * 70)
    print("  FINAL COMPARISON: AutoML vs Manual Training")
    print("=" * 70)
    
    comparison = pd.DataFrame({
        'Metric': ['Accuracy', 'F1-Score', 'Precision', 'Recall'],
        f'AutoML ({automl_model_name})': [automl_acc, automl_f1, automl_prec, automl_rec],
        'Manual RF': [rf_acc, rf_f1, rf_prec, rf_rec],
        'Manual XGB': [xgb_acc, xgb_f1, xgb_prec, xgb_rec],
        'Manual LGBM': [lgbm_acc, lgbm_f1, lgbm_prec, lgbm_rec],
    })
    
    print(comparison.to_string(index=False, float_format='%.4f'))
    
    # Visual comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(4)
    w = 0.2
    
    ax.bar(x - 1.5*w, [automl_acc, automl_f1, automl_prec, automl_rec], w, label=f'AutoML ({automl_model_name})', color='#e74c3c')
    ax.bar(x - 0.5*w, [rf_acc, rf_f1, rf_prec, rf_rec], w, label='Manual RF', color='#2ecc71')
    ax.bar(x + 0.5*w, [xgb_acc, xgb_f1, xgb_prec, xgb_rec], w, label='Manual XGB', color='#3498db')
    ax.bar(x + 1.5*w, [lgbm_acc, lgbm_f1, lgbm_prec, lgbm_rec], w, label='Manual LGBM', color='#f39c12')
    
    ax.set_xticks(x)
    ax.set_xticklabels(['Accuracy', 'F1-Score', 'Precision', 'Recall'])
    ax.set_ylim(0.7, 1.0)
    ax.legend()
    ax.set_title('AutoML vs Manual Models — adult.csv', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score')
    
    plt.tight_layout()
    plt.show()
    
    # Verdict
    best_manual = max(rf_acc, xgb_acc, lgbm_acc)
    diff = abs(automl_acc - best_manual)
    
    print(f"\n{'=' * 70}")
    if diff < 0.02:
        print(f"  ✅ VERDICT: AutoML results are CONSISTENT with manual training!")
        print(f"     Accuracy difference: {diff:.4f} (< 2%)")
    elif automl_acc > best_manual:
        print(f"  ✅ AutoML performed BETTER than manual ({automl_acc:.4f} vs {best_manual:.4f})")
        print(f"     This is expected — AutoML may use advanced feature engineering")
    else:
        print(f"  ⚠️ Manual performed better ({best_manual:.4f} vs {automl_acc:.4f})")
        print(f"     Difference: {diff:.4f}")
    print(f"{'=' * 70}")
else:
    print("⚠️ Skipping comparison — no AutoML .pkl available")
    print("   Your manual results are:")
    print(f"   Random Forest: {rf_acc:.4f}")
    print(f"   XGBoost:       {xgb_acc:.4f}")
    print(f"   LightGBM:      {lgbm_acc:.4f}")

## 1️⃣1️⃣ Sample Predictions

In [ ]:
# Show sample predictions from best model
sample_size = 20
sample_idx = np.random.RandomState(42).choice(len(y_test), sample_size, replace=False)

sample_actual = target_encoder.inverse_transform(y_test[sample_idx])
sample_rf = target_encoder.inverse_transform(rf_pred[sample_idx])
sample_xgb = target_encoder.inverse_transform(xgb_pred[sample_idx])
sample_lgbm = target_encoder.inverse_transform(lgbm_pred[sample_idx])

sample_df = pd.DataFrame({
    'Actual': sample_actual,
    'RF_Pred': sample_rf,
    'RF_Match': ['✅' if a == p else '❌' for a, p in zip(sample_actual, sample_rf)],
    'XGB_Pred': sample_xgb,
    'XGB_Match': ['✅' if a == p else '❌' for a, p in zip(sample_actual, sample_xgb)],
    'LGBM_Pred': sample_lgbm,
    'LGBM_Match': ['✅' if a == p else '❌' for a, p in zip(sample_actual, sample_lgbm)],
})

print("Sample Predictions (20 random test samples):")
print(sample_df.to_string(index=False))

rf_correct = sum(1 for a, p in zip(sample_actual, sample_rf) if a == p)
xgb_correct = sum(1 for a, p in zip(sample_actual, sample_xgb) if a == p)
lgbm_correct = sum(1 for a, p in zip(sample_actual, sample_lgbm) if a == p)

print(f"\nSample accuracy: RF={rf_correct}/{sample_size}, XGB={xgb_correct}/{sample_size}, LGBM={lgbm_correct}/{sample_size}")

## ✅ Summary

| Step | What | How |
|------|------|-----|
| 1 | Load Data | `adult.csv` — 32,561 rows × 15 columns |
| 2 | Clean | Replace `?` → mode, remove duplicates, strip whitespace |
| 3 | Encode Target | `income`: `<=50K`→0, `>50K`→1 (LabelEncoder) |
| 4 | Feature Engineering | Numeric→RobustScaler, Binary→LabelEncode, Low-card→One-Hot, High-card→Label+Freq |
| 5 | Split | 80/20, stratified, random_state=42 |
| 6 | Train | RandomForest, XGBoost, LightGBM (same params as AutoML) |
| 7 | Evaluate | Accuracy, F1, Precision, Recall, AUC-ROC |
| 8 | Compare | AutoML .pkl vs Manual — side-by-side metrics |